In [0]:
# ============================================================
# RETAIL MEDALLION PIPELINE — GOLD LAYER
# Description: Reads Silver table and builds two
#              analytics-ready Gold tables:
#
#   1. gold_sales_by_region  — monthly revenue by region
#   2. gold_top_products     — top 10 products per category
#
# Gold tables are the final consumption layer — used by
# BI dashboards, analysts, and reporting tools.
# ============================================================

SILVER_TABLE_NAME  = "silver_orders"
GOLD_SALES_TABLE   = "gold_sales_by_region"
GOLD_PRODUCTS_TABLE = "gold_top_products"

print(" Config loaded.")
print(f"   Source  : {SILVER_TABLE_NAME}")
print(f"   Target 1: {GOLD_SALES_TABLE}")
print(f"   Target 2: {GOLD_PRODUCTS_TABLE}")

 Config loaded.
   Source  : silver_orders
   Target 1: gold_sales_by_region
   Target 2: gold_top_products


In [0]:
# ------------------------------------------------------------
# STEP 1: Read from Silver layer
# ------------------------------------------------------------

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df_silver = spark.sql(f"SELECT * FROM {SILVER_TABLE_NAME}")

print(f" Silver data loaded. Total records: {df_silver.count()}")

 Silver data loaded. Total records: 9994


In [0]:
# ------------------------------------------------------------
# STEP 2: Build Gold Table 1 — Sales by Region
# ------------------------------------------------------------
# Aggregates monthly sales metrics per region and category.
# This table powers regional performance dashboards.
#
# Metrics produced:
#   - total_orders      : number of orders
#   - total_revenue     : sum of sale_price
#   - total_profit      : sum of profit
#   - avg_profit_margin : average margin %
#   - total_units_sold  : sum of quantity

from pyspark.sql.functions import (
    year, month, date_format, round,
    sum as spark_sum, count, avg
)

df_sales_by_region = df_silver \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("year_month", date_format("order_date", "yyyy-MM")) \
    .groupBy("year", "month", "year_month", "region", "category") \
    .agg(
        count("order_id").alias("total_orders"),
        round(spark_sum("sale_price"), 2).alias("total_revenue"),
        round(spark_sum("profit"), 2).alias("total_profit"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin"),
        round(spark_sum("quantity"), 0).alias("total_units_sold")
    ) \
    .orderBy("year", "month", "region")

print(f"Sales by Region built. Records: {df_sales_by_region.count()}")
df_sales_by_region.display()

Sales by Region built. Records: 288


year,month,year_month,region,category,total_orders,total_revenue,total_profit,avg_profit_margin,total_units_sold
2022,1,2022-01,Central,Technology,16,2871.4,171.4,9.74,45
2022,1,2022-01,Central,Office Supplies,63,3103.3,163.3,2.45,249
2022,1,2022-01,Central,Furniture,26,8694.1,794.1,6.81,79
2022,1,2022-01,East,Furniture,26,7619.6,759.6,7.96,100
2022,1,2022-01,East,Technology,22,12516.1,1106.1,7.97,69
2022,1,2022-01,East,Office Supplies,70,8404.1,854.1,4.2,263
2022,1,2022-01,South,Office Supplies,34,3586.9,296.9,3.77,149
2022,1,2022-01,South,Furniture,14,4756.1,536.1,9.4,63
2022,1,2022-01,South,Technology,14,3985.4,245.4,7.39,41
2022,1,2022-01,West,Technology,29,11161.1,1111.1,5.28,114


In [0]:
# ------------------------------------------------------------
# STEP 3: Build Gold Table 2 — Top Products per Category
# ------------------------------------------------------------
# Ranks products within each category by total profit.
# Keeps top 10 per category for product performance analysis.
#
# Window function used for ranking within each category
# so we get top 10 per category, not top 10 overall.

from pyspark.sql.functions import rank
from pyspark.sql.window import Window

# Aggregate at product level
df_products = df_silver \
    .groupBy("product_id", "category", "sub_category") \
    .agg(
        count("order_id").alias("total_orders"),
        round(spark_sum("quantity"), 0).alias("total_units_sold"),
        round(spark_sum("sale_price"), 2).alias("total_revenue"),
        round(spark_sum("profit"), 2).alias("total_profit"),
        round(avg("profit_margin_pct"), 2).alias("avg_profit_margin")
    )

# Rank within each category by total profit
window = Window.partitionBy("category").orderBy(
    df_products["total_profit"].desc()
)

df_top_products = df_products \
    .withColumn("rank_in_category", rank().over(window)) \
    .filter("rank_in_category <= 10") \
    .orderBy("category", "rank_in_category")

print(f" Top Products built. Records: {df_top_products.count()}")
df_top_products.display()

 Top Products built. Records: 30


product_id,category,sub_category,total_orders,total_units_sold,total_revenue,total_profit,avg_profit_margin,rank_in_category
FUR-CH-10002024,Furniture,Chairs,8,39,21096.2,2246.2,9.97,1
FUR-BO-10004834,Furniture,Bookcases,5,24,15024.1,1614.1,11.54,2
FUR-TA-10000198,Furniture,Tables,5,27,9559.7,1229.7,12.56,3
FUR-CH-10001854,Furniture,Chairs,8,28,8466.2,1146.2,13.05,4
FUR-CH-10004287,Furniture,Chairs,13,53,11103.7,1023.7,10.21,5
FUR-BO-10002213,Furniture,Bookcases,10,42,12471.2,1021.2,10.44,6
FUR-CH-10000454,Furniture,Chairs,12,51,10289.0,1009.0,8.65,7
FUR-CH-10004063,Furniture,Chairs,8,35,8455.7,1005.7,11.85,8
FUR-TA-10001889,Furniture,Tables,7,33,9189.1,979.1,9.41,9
FUR-CH-10004495,Furniture,Chairs,8,26,6227.6,877.6,12.56,10


In [0]:
# ------------------------------------------------------------
# STEP 4: Write both Gold tables to Delta Lake
# ------------------------------------------------------------

# Gold Table 1 — Sales by Region
df_sales_by_region.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_SALES_TABLE)

print(f" {GOLD_SALES_TABLE} written.")

# Gold Table 2 — Top Products
df_top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_PRODUCTS_TABLE)

print(f" {GOLD_PRODUCTS_TABLE} written.")

 gold_sales_by_region written.
 gold_top_products written.


In [0]:
# ------------------------------------------------------------
# Export Gold tables to CSV for Tableau
# ------------------------------------------------------------

# Export Sales by Region
spark.sql("SELECT * FROM gold_sales_by_region") \
    .toPandas() \
    .to_csv("/Volumes/workspace/default/retail_raw_data/gold_sales_by_region.csv", index=False)

# Export Top Products
spark.sql("SELECT * FROM gold_top_products") \
    .toPandas() \
    .to_csv("/Volumes/workspace/default/retail_raw_data/gold_top_products.csv", index=False)

print("CSV files exported to Volume.")

CSV files exported to Volume.
